# Fase 2 ABSA: Automatic Labeling dengan Model TermA (IndoBERT Fine-tuned)

Notebook ini memuat model hasil fine-tuning dari notebook sebelumnya
(`absa-fine-tuning-indobert.ipynb`, folder output **`termA-model-final`**)
dan menyediakan fungsi labeling otomatis: masukkan satu kalimat teks mentah
-> dapat label BIO per token (aspek + sentimen) -> tampil di layar sekaligus
tersimpan ke file `.txt` (format `token<TAB>label`, dipisah tab betulan,
sama seperti format data training aslinya).

Contoh: input `"parkirannya sempit sisanya oke."` menghasilkan:

```
parkirannya	B-ASPECT-FASILITAS
sempit	B-SENTIMENT-NEGATIVE
sisanya	O
oke	O
.	O
```

**Penting soal `MODEL_DIR` (baca sebelum run):**
- Kalau notebook ini dijalankan **lanjut di sesi Kaggle yang sama** dengan
  training (belum restart kernel, cuma menambah sel baru di bawah): path
  bawaan `/kaggle/working/termA-model-final` sudah otomatis benar, tidak
  perlu diapa-apakan lagi.
- Kalau ini **notebook/sesi Kaggle terpisah** (kasus paling umum untuk
  "notebook baru"): folder `/kaggle/working/` mulai kosong lagi karena
  scoped per-sesi. Langkah yang perlu dilakukan:
  1. Di notebook training, buka tab **Output** -> cari folder
     `termA-model-final` -> klik **"New Dataset"** untuk mem-publish folder
     itu jadi Kaggle Dataset.
  2. Di notebook ini, klik **Add Input** -> pilih dataset yang baru dibuat.
  3. Sel konfigurasi di bawah sudah mencoba menebak beberapa path umum
     secara otomatis; kalau tidak ketemu, isi manual variabel `MODEL_DIR`
     sesuai path dataset Anda (biasanya
     `/kaggle/input/<nama-dataset>/termA-model-final`).
- Di luar Kaggle (Colab/lokal): isi `MODEL_DIR` manual ke lokasi folder
  model Anda.

Skema label (`id2label`) dibaca otomatis dari `config.json` di dalam
`MODEL_DIR`, bukan didefinisikan ulang manual di sini -- supaya notebook
ini selalu konsisten dengan `LABEL_LIST` asli di notebook training, persis
prinsip yang sama dipakai `AspectOpinionExtractor` pada notebook training
(lihat Bagian 3 di sana).

In [1]:
import os
import re
import html
import unicodedata
from typing import List, Tuple

import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

In [2]:
# Load model hasil train
MODEL_DIR = "/kaggle/input/notebooks/dbernardons/absa-fine-tuning-indobert/termA-model-final"

In [3]:
# Disalin PERSIS dari Bagian 3 notebook training (absa-fine-tuning-indobert.ipynb)
# -- WAJIB identik supaya kontrak tokenisasi input di sini sama dengan kontrak
# yang dipakai model saat training (tokenize_and_align_labels di Bagian 2).
_HTML_TAG_RE = re.compile(r"<[^>]+>")
_WHITESPACE_RE = re.compile(r"\s+")
_TOKEN_RE = re.compile(r"\w+(?:[-']\w+)*|[^\w\s]")  # kata (boleh ada - atau ' di tengah) ATAU 1 tanda baca


def clean_text(text: str) -> str:
    """Encoding rusak, artefak HTML sisa scraping, whitespace berlebih.
    TIDAK stemming/stopword-removal/lowercasing paksa -- itu merusak span
    verbatim yang jadi kontrak dasar model token-classification ini."""
    text = html.unescape(text)
    text = _HTML_TAG_RE.sub(" ", text)
    text = unicodedata.normalize("NFKC", text)
    text = text.encode("utf-8", "ignore").decode("utf-8")
    text = _WHITESPACE_RE.sub(" ", text).strip()
    return text


def tokenize(text: str) -> List[str]:
    """Pisahkan kata dan tanda baca jadi token terpisah -- koma/titik jadi
    token sendiri, bukan menempel ke kata sebelumnya."""
    return _TOKEN_RE.findall(text)

In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu"
MAX_LENGTH = 256  # sama dengan max_length saat training (Bagian 2 & 3 notebook training)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()
if device == "cuda":
    model = model.half()

# id2label diambil dari config checkpoint (bukan ditulis ulang manual di sini) --
# supaya otomatis konsisten dengan LABEL_LIST asli di notebook training, persis
# prinsip yang sama dipakai AspectOpinionExtractor pada notebook training.
id2label = {int(k): v for k, v in model.config.id2label.items()}

print(f"Model siap. device={device}, jumlah label={len(id2label)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model siap. device=cuda, jumlah label=27


In [5]:
@torch.inference_mode()
def label_tokens(tokens: List[str]) -> List[str]:
    """Beri label BIO ke tiap token (hasil tokenize()) memakai termA-model-final.
    Strategi alignment word-piece: subword PERTAMA tiap kata yang menentukan
    label kata itu -- konsisten dengan cara label dipasangkan saat training
    (tokenize_and_align_labels di Bagian 2 notebook training)."""
    if not tokens:
        return []

    enc = tokenizer(
        [tokens], is_split_into_words=True, truncation=True,
        max_length=MAX_LENGTH, padding=True, return_tensors="pt",
    ).to(device)

    pred_ids = model(**enc).logits.argmax(-1)[0].tolist()
    word_ids = enc.word_ids(batch_index=0)

    labels, prev_w, max_word_seen = [], None, -1
    for wid, pid in zip(word_ids, pred_ids):
        if wid is None or wid == prev_w:
            continue
        labels.append(id2label[pid])
        prev_w = wid
        max_word_seen = wid

    # Peringatan kalau input ke-truncate di max_length (sisa token otomatis "O")
    n_covered = max_word_seen + 1
    if n_covered < len(tokens):
        print(f"[PERINGATAN] Input terpotong di max_length={MAX_LENGTH}: "
              f"{len(tokens)} token asli, cuma {n_covered} tercakup model. "
              f"{len(tokens) - n_covered} token terakhir otomatis diberi label 'O'.")

    return (labels + ["O"] * len(tokens))[:len(tokens)]


def label_text(text: str) -> List[Tuple[str, str]]:
    """Terima 1 kalimat teks mentah, kembalikan list (token, label)."""
    tokens = tokenize(clean_text(text))
    labels = label_tokens(tokens)
    return list(zip(tokens, labels))

In [6]:
def save_bio_txt(sentences_labeled: List[List[Tuple[str, str]]], output_path: str, mode: str = "w") -> None:
    """Simpan hasil labeling ke .txt: format `token<TAB>label` per baris, 1 baris
    kosong sebagai pemisah antar kalimat -- sama dengan format file data training
    asli (hotel_preprocess_labeled_train.txt), jadi bisa dibaca ulang oleh
    load_iob_file() dari notebook training kalau suatu saat dibutuhkan."""
    with open(output_path, mode, encoding="utf-8") as f:
        for i, sent in enumerate(sentences_labeled):
            for token, label in sent:
                f.write(f"{token}\t{label}\n")
            if i != len(sentences_labeled) - 1:
                f.write("\n")


def label_and_save(text: str, output_path: str = "/kaggle/working/hasil_labeling.txt",
                    mode: str = "w", verbose: bool = True) -> List[Tuple[str, str]]:
    """Fungsi labeling otomatis end-to-end: teks mentah masuk -> BIO label keluar,
    dicetak ke layar (dipisah TAB) sekaligus tersimpan ke output_path."""
    hasil = label_text(text)
    if verbose:
        for token, label in hasil:
            print(f"{token}\t{label}")
    save_bio_txt([hasil], output_path, mode=mode)
    return hasil

In [7]:
import pandas as pd

# Baca dataset
CSV_PATH = "/kaggle/input/datasets/dbernardons/dataset-maribaya-fake/dataset maribaya for labeling.csv"
df = pd.read_csv(CSV_PATH)

OUTPUT_PATH = "/kaggle/working/hasil_labeling.txt"

# Mulai file baru
first = True

for ulasan in df["ulasan"].fillna("").astype(str):

    label_and_save(
        text=ulasan,
        output_path=OUTPUT_PATH,
        mode="w" if first else "a",   # pertama write, berikutnya append
        verbose=False                 # tidak print semua hasil ke notebook
    )

    first = False

print(f"Hasil tersimpan di: {OUTPUT_PATH}")

Hasil tersimpan di: /kaggle/working/hasil_labeling.txt
